# CrewAI Web Research Agent Demo

This notebook demonstrates how to build a web research agent using CrewAI with custom tools for web scraping and text analysis. The agent is instrumented with Langfuse for observability and tracing.

## Overview
- Create custom tools for web scraping and text analysis
- Set up a CrewAI agent with specialized capabilities
- Execute research tasks with automatic instrumentation and logging


In [1]:
%pip install crewai crewai-tools requests beautifulsoup4 python-dotenv langfuse openinference-instrumentation-crewai opentelemetry-exporter-otlp -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import requests
from bs4 import BeautifulSoup
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langfuse import get_client
from openinference.instrumentation.crewai import CrewAIInstrumentor
from openinference.instrumentation.litellm import LiteLLMInstrumentor

# Load environment variables from .env file
load_dotenv()


ImportError: cannot import name 'get_client' from partially initialized module 'langfuse' (most likely due to a circular import) (/home/kiran/Flotorch-fork/flotorch-eval/langfuse.py)

In [ ]:
# Set up environment variables:
# OPENAI_API_KEY
# LANGFUSE_SECRET_KEY  
# LANGFUSE_HOST
# LANGFUSE_PUBLIC_KEY

# Initialize Langfuse client
try:
    langfuse = get_client()
    
    if langfuse.auth_check():
        print("Langfuse client authenticated and ready!")
    else:
        print("Running in local mode (no cloud connection)")
        
except Exception as e:
    print(f"Langfuse setup: {e}")
    print("Continuing in local mode...")
    langfuse = get_client()

print("Setting up CrewAI auto-instrumentation...")
CrewAIInstrumentor().instrument(skip_dep_check=True)
LiteLLMInstrumentor().instrument()

Langfuse client authenticated and ready!
Setting up CrewAI auto-instrumentation...


## Custom Tool Implementations

Define specialized tools that agents can use to perform web scraping and text analysis tasks.


### Web Scraper Tool

Extracts and cleans text content from web pages with proper error handling.


In [4]:
# Define input schema for web scraper tool
class WebScraperToolInput(BaseModel):
    """Input schema for WebScraperTool."""
    url: str = Field(..., description="The URL to scrape content from")

class WebScraperTool(BaseTool):
    """Tool for scraping and cleaning web content."""
    name: str = "Web Scraper"
    description: str = "Scrapes content from a given URL and returns cleaned text content"
    args_schema: Type[BaseModel] = WebScraperToolInput

    def _run(self, url: str) -> str:
        """
        Scrape content from the given URL and return cleaned text.
        
        Args:
            url: The website URL to scrape
            
        Returns:
            Cleaned text content from the webpage (max 3000 chars)
        """
        try:
            # Set browser-like headers to avoid bot detection
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
            
            # Fetch webpage with timeout
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()  # Raise exception for HTTP errors
            
            # Parse HTML content
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Remove script and style elements that contain no useful text
            for script in soup(["script", "style"]):
                script.decompose()
            
            # Extract text content
            text = soup.get_text()
            
            # Clean up whitespace and formatting
            lines = (line.strip() for line in text.splitlines())
            chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
            text = '\n'.join(chunk for chunk in chunks if chunk)
            
            # Limit text length to prevent overwhelming the agent
            return text[:3000] + "..." if len(text) > 3000 else text
            
        except Exception as e:
            return f"Error scraping URL: {str(e)}"

print("Web Scraper Tool created!")


Web Scraper Tool created!


### Text Analyzer Tool

Analyzes text content and provides statistical insights with keyword extraction.


In [5]:
# Define input schema for text analyzer tool
class TextAnalyzerToolInput(BaseModel):
    """Input schema for TextAnalyzerTool."""
    text: str = Field(..., description="The text to analyze")
    include_keywords: bool = Field(default=True, description="Include top keywords in analysis")

class TextAnalyzerTool(BaseTool):
    """Tool for analyzing text content and extracting statistical insights."""
    name: str = "Text Analyzer"
    description: str = "Analyzes text and provides key statistics and insights with keyword extraction"
    args_schema: Type[BaseModel] = TextAnalyzerToolInput

    def _run(self, text: str, include_keywords: bool = True) -> str:
        """
        Analyze the given text and return comprehensive statistics.
        
        Args:
            text: The text content to analyze
            include_keywords: Whether to extract top keywords (default: True)
            
        Returns:
            Formatted analysis report with statistics and optional keywords
        """
        try:
            # Calculate basic text statistics
            words = text.split()
            word_count = len(words)
            char_count = len(text)
            sentence_count = len([s for s in text.split('.') if s.strip()])
            avg_word_length = sum(len(word) for word in words) / len(words) if words else 0
            
            # Build comprehensive analysis report
            analysis = f"""Text Analysis Summary:
• Word Count: {word_count:,}
• Character Count: {char_count:,} 
• Sentence Count: {sentence_count:,}
• Average Word Length: {avg_word_length:.1f} characters"""
            
            # Extract keywords if requested
            if include_keywords and words:
                # Find top 5 most common meaningful words
                clean_words = []
                for word in words:
                    clean_word = word.lower().strip('.,!?;:"()[]{}').replace("'", "")
                    if len(clean_word) > 3 and clean_word.isalpha():
                        clean_words.append(clean_word)
                
                if clean_words:
                    # Calculate word frequency
                    word_freq = {}
                    for word in clean_words:
                        word_freq[word] = word_freq.get(word, 0) + 1
                    
                    # Get top 5 most frequent words
                    top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:5]
                    keywords = ", ".join([word for word, count in top_words])
                    analysis += f"\n• Top Keywords: {keywords}"
            
            return analysis
            
        except Exception as e:
            return f"Error analyzing text: {str(e)}"

print("Text Analyzer Tool created!")


Text Analyzer Tool created!


## Agent Setup

Initialize tools and create a specialized research agent with custom capabilities.


In [6]:
# Initialize custom tool instances
web_scraper = WebScraperTool()
text_analyzer = TextAnalyzerTool()

# Create specialized research agent with custom tools
research_agent = Agent(
    role='Web Research Analyst',
    goal='Scrape web content and analyze it to provide comprehensive insights',
    backstory="""You are an expert web research analyst with the ability to scrape 
    content from websites and analyze text data. You excel at extracting meaningful 
    information from web pages and providing detailed analysis of textual content.""",
    verbose=True,
    allow_delegation=False,
    tools=[web_scraper, text_analyzer]
)

print("Research Agent created with 2 custom tools!")


Research Agent created with 2 custom tools!


## Task Definition and Execution

Define a research task and create a crew to execute it with automatic instrumentation.


In [7]:
# Define comprehensive research task with clear instructions
research_task = Task(
    description="""
    Research and analyze content from https://www.flotorch.ai/blogs/crewai-financial-agent-with-flotorch website:
    
    1. Use the Web Scraper tool to extract content from the website
    2. Use the Text Analyzer tool to analyze the scraped content
    3. Provide a comprehensive summary that includes:
       - Key information found on the website
       - Text analysis statistics (word count, character count, etc.)
       - Top keywords identified
       - Main insights and takeaways from the content
    
    Make sure to use both tools in sequence and provide detailed analysis.
    """,
    agent=research_agent,  # Assign to our research agent
    expected_output="A comprehensive report including website content summary and detailed text analysis statistics"
)

# Create crew with single agent and task
research_crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    verbose=False,
    process=Process.sequential
)

print("Research task and crew configured successfully!")


Research task and crew configured successfully!


## Execute Research

Run the crew to perform the web research task with full observability tracking.


In [8]:
print("Starting CrewAI agent with Auto-Instrumentation...")
print("=" * 60)

result = research_crew.kickoff()
print("\nFinal Result:")
print(result)

# Ensure all traces are sent to Langfuse
langfuse.flush()

Starting CrewAI agent with Auto-Instrumentation...


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Research Analyst                                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Research and analyze content from https://www.flotorch.ai/blogs/crewai-financial-agent-with-flotorch       │
│  website:                                                                                                       │
│                                                                                                                 │
│      1. Use the Web Scraper tool to extract content from the website                                            │
│      2. Use the Text Analyzer tool to analyze the scraped content                                               │
│      3. Provide a comprehensive summary that includes:                                                          │
│         - Key information found on the website                                                                  │
│         - Text analysis statistics (word count, character count, etc.)                                          │
│         - Top keywords identified                                                                               │
│         - Main insights and takeaways from the content                                                          │
│                                                                                                                 │
│      Make sure to use both tools in sequence and provide detailed analysis.                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Research Analyst                                                                                    │
│                                                                                                                 │
│  Thought: Action: Web Scraper                                                                                   │
│                                                                                                                 │
│  Using Tool: Web Scraper                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"url\": \"https://www.flotorch.ai/blogs/crewai-financial-agent-with-flotorch\"}"                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flotorch                                                                                                       │
│  Github LinkProductLLMOps & FMOps OptimisationRAG PipelinePerformance, Cost & SecurityLLM                       │
│  RoutingRecommendation EngineNocode Workflow ManagementUnified Gateway ValuePrompt ManagementCache              │
│  ManagementEnterprise Security & GuardrailsReal-Time Performance & Usage AnalyticsCompanyEventsNews             │
│  RoomCareersContact usResourcesblogsGlossaryFAQsGithubSchedule DemoHomeBlogArticleCrewAI Financial Agent with   │
│  FloTorchDownload🧠 Automate Your Finances with AI Agents: A Deep Dive into crewai-finagentManaging personal    │
│  finances shouldn’t feel like a full-time job. But sifting through bank statements, categorizing expenses, and  │
│  identifying trends can be time-consuming and overwhelming.What if AI could do all of that for you?Meet         │
│  crewai-finagent — an application that uses a team of AI agents to turn your PDF bank statements into           │
│  structured data, categorized expenses, and actionable financial insights. Upload your PDF and watch the        │
│  agents go to work.📽️ Watch it in action: Check out the demo video below to see how crewai-finagent simplifies   │
│  personal finance management.In this post, we’ll walk through how it works, how to get started, and where it    │
│  fits in the future of AI-powered personal finance.🚀 What is crewai-finagent?crewai-finagent is an AI-powered  │
│  application developed using FloTorch, a modern agent orchestration and experimentation framework. It           │
│  leverages the CrewAI ecosystem to build multi-agent LLM workflows where specialized agents collaborate to      │
│  complete a task.In this case, the goal is simple but powerful: take a financial PDF, and turn it into          │
│  structured, categorized, and insightful data — automatically.In this project, three AI agents work together    │
│  in sequence.Extract raw transactions from a PDF.Categorize those transactions by type (e.g., groceries,        │
│  rent).Analyze the data to produce a summary of your financial habits.All of this happens without any manual    │
│  tagging, spreadsheets, or categorization rules — it’s AI-powered from start to finish.🛠 Installation &         │
│  SetupHere...                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Research Analyst                                                                                    │
│                                                                                                                 │
│  Thought: Action: Text Analyzer                                                                                 │
│                                                                                                                 │
│  Using Tool: Text Analyzer                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"text\": \"Flotorch\\nGithub LinkProductLLMOps & FMOps OptimisationRAG PipelinePerformance, Cost &          │
│  SecurityLLM RoutingRecommendation EngineNocode Workflow ManagementUnified Gateway ValuePrompt ManagementCache  │
│  ManagementEnterprise Security & GuardrailsReal-Time Performance & Usage AnalyticsCompanyEventsNews             │
│  RoomCareersContact usResourcesblogsGlossaryFAQsGithubSchedule DemoHomeBlogArticleCrewAI Financial Agent with   │
│  FloTorchDownload\\ud83e\\udde0 Automate Your Finances with AI Agents: A Deep Dive into                         │
│  crewai-finagentManaging personal finances shouldn\\u2019t feel like a full-time job. But sifting through bank  │
│  statements, categorizing expenses, and identifying trends can be time-consuming and overwhelming.What if AI    │
│  could do all of that for you?Meet crewai-finagent \\u2014 an application that uses a team of AI agents to      │
│  turn your PDF bank statements into structured data, categorized expenses, and actionable financial insights.   │
│  Upload your PDF and watch the agents go to work.\\ud83d\\udcfd\\ufe0f Watch it in action: Check out the demo   │
│  video below to see how crewai-finagent simplifies personal finance management.In this post, we\\u2019ll walk   │
│  through how it works, how to get started, and where it fits in the future of AI-powered personal               │
│  finance.\\ud83d\\ude80 What is crewai-finagent?crewai-finagent is an AI-powered application developed using    │
│  FloTorch, a modern agent orchestration and experimentation framework. It leverages the CrewAI ecosystem to     │
│  build multi-agent LLM workflows where specialized agents collaborate to complete a task.In this case, the      │
│  goal is simple but powerful: take a financial PDF, and turn it into structured, categorized, and insightful    │
│  data \\u2014 automatically.In this project, three AI agents work together in sequence.Extract raw              │
│  transactions from a PDF.Categorize those transactions by type (e.g., groceries, rent).Analyze the data to      │
│  produce a summary of your financial habits.All of this happens without any manual tagging, spreadsheets, or    │
│  categorization rules \\u2014 it\\u2019s AI-powered from start to finish.\\ud83d\\udee0 Installation &          │
│  SetupHere\\u2019s how to get started:\\u2705 RequirementsPython 3.10+FloTorch Base URL and\\u00a0 API key1.    │
│  Clone the repository\\n# Run this in your terminal\\n# git clone                                               │
│  https://github.com/FissionAI/flotorch-labs.git\\n# cd finance/crewai-finance-agents/\\n2. Create and activate  │
│  a virtual environment\\n# python -m venv venv\\n# source venv/bin/activate (For Linux/macOS)\\n#               │
│  venv\\\\Scripts\\\\activate(For Windows)\\n3. Install dependencies\\n# pip install                             │
│  -rsrc/financial_document_analyzer/requirements.txt\\n4. Set up environment variables\\n(For Linux/macOS)\\n#   │
│  export OPENAI_BASE_URL=https://<gateway-url>/api/openai/v1\\n# export OPENAI_API_KEY=<secret-key>\\n(For       │
│  Windows)\\n# set OPENAI_BASE_URL=https://<gateway-url>/api/openai/v1\\n# set                                   │
│  OPENAI_API_KEY=<secret-key>\\n(or create .env file with)\\n#                                                   │
│  OPENAI_BASE_URL=https://<gateway-url>/api/openai/v1\\n# OPENAI_API_KEY=<secret-key>\\n5. Run App\\n# python    │
│  -m streamlit run src/financial_document_analyzer/streamlit_app/app.pyYou\\u2019ll be prompted to upload a      │
│  financial PDF \\u2014 for example, a bank or credit card statement...\", \"include_keywords\": true}"          │
│                                                       

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Text Analysis Summary:                                                                                         │
│  • Word Count: 371                                                                                              │
│  • Character Count: 3,008                                                                                       │
│  • Sentence Count: 28                                                                                           │
│  • Average Word Length: 7.1 characters                                                                          │
│  • Top Keywords: financial, your, agents, this, with                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Research Analyst                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Flotorch                                                                                                       │
│  Github LinkProductLLMOps & FMOps OptimisationRAG PipelinePerformance, Cost & SecurityLLM                       │
│  RoutingRecommendation EngineNocode Workflow ManagementUnified Gateway ValuePrompt ManagementCache              │
│  ManagementEnterprise Security & GuardrailsReal-Time Performance & Usage AnalyticsCompanyEventsNews             │
│  RoomCareersContact usResourcesblogsGlossaryFAQsGithubSchedule DemoHomeBlogArticleCrewAI Financial Agent with   │
│  FloTorchDownload🧠 Automate Your Finances with AI Agents: A Deep Dive into crewai-finagentManaging personal    │
│  finances shouldn’t feel like a full-time job. But sifting through bank statements, categorizing expenses, and  │
│  identifying trends can be time-consuming and overwhelming.What if AI could do all of that for you?Meet         │
│  crewai-finagent — an application that uses a team of AI agents to turn your PDF bank statements into           │
│  structured data, categorized expenses, and actionable financial insights. Upload your PDF and watch the        │
│  agents go to work.📽️ Watch it in action: Check out the demo video below to see how crewai-finagent simplifies   │
│  personal finance management.In this post, we’ll walk through how it works, how to get started, and where it    │
│  fits in the future of AI-powered personal finance.🚀 What is crewai-finagent?crewai-finagent is an AI-powered  │
│  application developed using FloTorch, a modern agent orchestration and experimentation framework. It           │
│  leverages the CrewAI ecosystem to build multi-agent LLM workflows where specialized agents collaborate to      │
│  complete a task.In this case, the goal is simple but powerful: take a financial PDF, and turn it into          │
│  structured, categorized, and insightful data — automatically.In this project, three AI agents work together    │
│  in sequence.Extract raw transactions from a PDF.Categorize those transactions by type (e.g., groceries,        │
│  rent).Analyze the data to produce a summary of your financial habits.All of this happens without any manual    │
│  tagging, spreadsheets, or categorization rules — it’s AI-powered from start to finish.🛠 Installation &         │
│  SetupHere’s how to get started:✅ RequirementsPython 3.10+FloTorch Base URL and API key1. Clone the            │
│  repository                                                                                                     │
│  # Run this in your terminal                                                                                    │
│  # git clone https://github.com/FissionAI/flotorch-labs.git                                                     │
│  # cd finance/crewai-finance-agents/                                                                            │
│  2. Create and activate a virtual environment                                                                   │
│  # python -m venv venv                                                                                          │
│  # source venv/bin/activate (For Linux/macOS)                                                                   │
│  # venv\Scripts\activate(For Windows)                                                                           │
│  3. Install dependencies                                 


Final Result:
Flotorch
Github LinkProductLLMOps & FMOps OptimisationRAG PipelinePerformance, Cost & SecurityLLM RoutingRecommendation EngineNocode Workflow ManagementUnified Gateway ValuePrompt ManagementCache ManagementEnterprise Security & GuardrailsReal-Time Performance & Usage AnalyticsCompanyEventsNews RoomCareersContact usResourcesblogsGlossaryFAQsGithubSchedule DemoHomeBlogArticleCrewAI Financial Agent with FloTorchDownload🧠 Automate Your Finances with AI Agents: A Deep Dive into crewai-finagentManaging personal finances shouldn’t feel like a full-time job. But sifting through bank statements, categorizing expenses, and identifying trends can be time-consuming and overwhelming.What if AI could do all of that for you?Meet crewai-finagent — an application that uses a team of AI agents to turn your PDF bank statements into structured data, categorized expenses, and actionable financial insights. Upload your PDF and watch the agents go to work.📽️ Watch it in action: Check out the d